# Testing and validation

The test suite checks numerical correctness and agreement with R:

1. **Internal correctness** — analytic gradients match finite differences;
   offsets are true risk minimizers; learners reproduce their known closed
   forms. These tests run without R and live in the Rust core.
2. **R parity** — the full fitted path matches R `mboost` / `gamboostLSS` on
   the same data, to tight tolerances. These tests run only when R and `rpy2`
   are available, and are skipped by default.

The two layers protect against different failure modes: internal tests catch
math errors in new code; parity tests catch subtle interactions and confirm
that the documented R correspondence still holds.

## Test layout

``````
tests/                                    # Python (pytest)
|-- conftest.py                           # r_parity marker; HAS_R / HAS_RPY2 flags
|-- test_smoke.py                         # import + tiny-fit smoke (no R)
|-- test_property.py                      # property-based (hypothesis)
|-- test_sklearn_api.py                   # sklearn estimator checks
|-- test_lss_smoke.py
|-- test_engine/                          # engine-level unit tests
|-- test_families/                        # per-family unit tests
|-- test_learners/                        # per-learner unit tests
|-- test_glmboost_parity/                 # $\leftarrow$ R parity, one file per family
|   |-- test_gaussian_two_predictors.py
|   |-- test_binomial_bols.py
|   |-- test_poisson_bols.py
|   +-- ... (16 family-specific files)
|-- test_gamboost_parity/                 # $\leftarrow$ R parity for gamboost
|-- test_gamlss_parity/                   # $\leftarrow$ R parity for LSS families
|-- test_cvrisk_parity/                   # $\leftarrow$ cvrisk vs R
|-- test_inference_parity/                # coef / confint / varimp vs R
|-- test_multinomial_parity/
|-- test_predict_parity/
|-- test_stabsel_parity/
+-- fixtures/                             # shared R-generated reference data

crates/mboostrs-core/                     # Rust unit tests (#[test])
+-- src/
    |-- families.rs      # inline #[cfg(test)] mod tests
    |-- learners.rs      # inline
    |-- lss.rs           # inline (incl. finite-diff helper)
    |-- engine.rs        # inline
    +-- ...
+-- tests/
    +-- lognormal_r_parity.rs             # integration test
``````

## Running the tests

``````bash
# Rust unit tests (always available)
cargo test --workspace

# Python tests without R (CI default)
pytest -m "not r_parity"

# Full suite including parity (requires R + rpy2 locally)
pytest --run-r-parity

# The R package statically links the Rust core at install time
# (mboostrs/src/Makevars, -force_load). After any change to
# crates/mboostrs-core or crates/mboostrs-r, rebuild it before the
# parity suite, or the tests fail with a "wrap__r_<symbol>" not
# available error that reads like a code bug but is a stale build:
cargo build -p mboostrs-r --release && R CMD INSTALL mboostrs
``````

## The `r_parity` marker

The marker mechanism (`tests/conftest.py`) is the gate that keeps CI green
without R while letting local developers run the full suite:

``````python
def pytest_configure(config):
    config.addinivalue_line("markers", "r_parity: requires R + rpy2")

def pytest_collection_modifyitems(config, items):
    if config.getoption("--run-r-parity"):
        return
    skip_r = pytest.mark.skip(reason="need --run-r-parity option to run")
    for item in items:
        if "r_parity" in item.keywords:
            item.add_marker(skip_r)
``````

Two environment flags are also exposed:

``````python
HAS_RPY2 = importlib.util.find_spec("rpy2") is not None
HAS_R = <subprocess R --version succeeds>
``````

A test that needs R should be decorated `@pytest.mark.r_parity` and should
also `pytest.skip` explicitly when `HAS_R` or `HAS_RPY2` is false, so the
skip reason is informative even if the marker is bypassed.

## Anatomy of a parity test

Every parity test follows the same shape: generate data once (in Python or
R), fit in both engines on the **same** data, assert agreement on the
quantities that define a fit. The template below is
`tests/test_glmboost_parity/test_gaussian_two_predictors.py`:

``````python
@pytest.mark.r_parity
def test_gaussian_two_predictors_parity():
    mstop_val, nu_val = 50, 0.1
    r = _run_r_glmboost(mstop_val, nu_val)            # R via rpy2
    fit = mboost.glmboost(
        "y ~ x1 + x2",
        data={"x1": r["x1"], "x2": r["x2"], "y": r["y"]},
        family=Gaussian(),
        control=mboost.boost_control(mstop=mstop_val, nu=nu_val),
    )
    rtol = 1e-10
    np.testing.assert_allclose(fit.fitted_, r["fitted"], rtol=rtol)
    np.testing.assert_allclose(fit.risk_,   r["risk"],   rtol=rtol)
    np.testing.assert_allclose(fit.offset_, r["offset"], rtol=rtol)
    # coefficients compared by name ...
``````

The quantities asserted are exactly those documented in
[](parity_and_bugs.ipynb): the risk path (length $m_{\text{stop}}+1$), the
fitted values, the selection sequence, the coefficients, and the offset.

### Tolerance bands

Tolerances are set per test module, next to the assertions, and justified by
the actual source of divergence from R (not picked by guess). The guiding table gives the relative tolerance (`rtol`, the largest allowed |x - y| / |y| between the two results):

| Test category | rtol | Justification |
|---|---|---|
| `gaussian_bols` | $10^{-10}$ | Pure linear algebra; only floating-point path divergence. |
| `bbs_default` | $10^{-8}$ | Penalty Cholesky solve path differs slightly from R. |
| `bmono` | $10^{-6}$ | Constrained solver iterates to a slightly different active set. |
| LSS survival offsets | $10^{-6}$ | Brent vs R `optimize()` for the nuisance parameter. |
| Multinomial | $10^{-6}$ | End-to-end; isolated components match to $10^{-11}$. |
| Offsets / gradients | $10^{-12}$–$10^{-14}$ | Same bracket and tolerance as R's `optimize`. |

If you add a parity test, set the tolerance from observed divergence, then
document the source. Picking `$10^{-6}$ everywhere` hides real bugs.

## Internal tests: the finite-difference gradient checker

The primary internal test is the finite-difference gradient
verifier, `assert_ngradient_matches_finite_diff` (`crates/mboostrs-core/src/lss.rs:3254`).
It confirms that each analytic negative gradient is the exact central
derivative of the risk:

``````rust
fn assert_ngradient_matches_finite_diff(
    fam: &dyn LssFamily, y: &[f64], etas: &mut Vec<Array1<f64>>, n: usize,
) {
    let w = vec![1.0; n];
    let h = 1e-6;
    for k in 0..fam.n_params() {
        let mut ngr = vec![0.0; n];
        fam.ngradient(k, y, etas, &w, &mut ngr);
        for j in 0..n {
            let orig = etas[k][j];
            etas[k][j] = orig + h;  let rp = fam.risk(y, etas, &w);
            etas[k][j] = orig - h;  let rm = fam.risk(y, etas, &w);
            etas[k][j] = orig;
            let fd = -(rp - rm) / (2.0 * h);          // -d(risk)/d(eta)
            assert!(
                (ngr[j] - fd).abs() < 1e-5 * (1.0 + ngr[j].abs()),
                "{} param {k} obs {j}: ngradient {} != finite-diff {}",
                fam.name(), ngr[j], fd,
            );
        }
    }
}
``````

This single helper catches **both** gradient errors and risk-sign errors
simultaneously. It is invoked from at least one `#[test]` per LSS family —
e.g. `zinb_and_zipo_ngradients_match_finite_diff` (`lss.rs:3285`) covers the
zero-inflated families whose gradients are intricate.

``````{admonition} When you add an LSS family
:class: important
Always add a `#[test]` that calls `assert_ngradient_matches_finite_diff` on
your new family. Without it, sign errors and chain-rule mistakes can hide
behind an implementation that is internally consistent but mathematically
wrong. The helper catches gradient sign errors and chain-rule errors in the analytic gradient.
``````

The single-parameter families have an analogous check in
`crates/mboostrs-core/src/families.rs`. The principle is the same: the
analytic ngradient must equal $-d(\text{risk})/d\eta$ to within $10^{-5}$
relative.

## Internal tests: golden values and closed forms

Beyond finite differences, the Rust core carries `#[test]`s that pin known
closed forms:

- **Gaussian offset** equals the weighted mean of $y$.
- **Laplace / Quantile offset** equals R's type-7 weighted quantile
  (`families.rs:94-113`).
- **`bbs` basis dimensions**: $2(\text{degree}+1) + \text{n\_inner}$ knots,
  $\text{n\_inner} + \text{degree} + 1$ columns (`learners.rs:2171`).
- **`df_to_lambda`** round-trips: solving for $\lambda$ at target `df`, then
  evaluating `df(λ)`, recovers the target to $10^{-12}$.
- **`KroneckerBols`** per-class penalty is exactly $I + K_1$ (the `+I` leak
  from R's `%O%`; see [](kronecker_multinomial.ipynb)).

These tests are independent of R and run on every `cargo test`. They are the
first line of defence against regressions in the numerics.

## Invariant tests

`tests/test_property.py` checks several invariants against fixed, hand-picked
examples (one seed, one shape per test):

- Fitting twice on the same data gives bit-identical paths (determinism).
- `predict(newdata=training_data)` reproduces `fitted_`.
- Truncation `fit[:k].fitted_` equals `fitted_` at iteration $k$.
- Risk is monotonically non-increasing along the path (Gaussian).

Most of that file is example-based, not property-based: each test picks one
seed and one data shape, so it checks the invariant on a single case rather
than searching for a counterexample across the input space. One test,
`test_glmboost_risk_nonincreasing_property`, uses `hypothesis` to generate
`(seed, n, p, mstop, nu)` tuples and check the monotone-risk property on each —
the only property-based test in that file. The worked example in the Example property-based test subsection is the same check, run live.

### Example property-based test



In [1]:
from hypothesis import given, settings, strategies as st
import numpy as np
import mboostrs as mb

failures = []


@given(
    seed=st.integers(min_value=0, max_value=2**31 - 1),
    n=st.integers(min_value=20, max_value=100),
    p=st.integers(min_value=1, max_value=5),
    mstop=st.integers(min_value=5, max_value=100),
    nu=st.floats(min_value=0.05, max_value=0.3),
)
@settings(max_examples=30, deadline=None)
def check_risk_nonincreasing(seed, n, p, mstop, nu):
    rng = np.random.default_rng(seed)
    X = rng.normal(size=(n, p))
    beta = rng.normal(scale=1.0, size=p)
    y = X @ beta + rng.normal(scale=0.5, size=n)
    data = {f"x{j}": X[:, j] for j in range(p)}
    data["y"] = y
    formula = "y ~ " + " + ".join(f"x{j}" for j in range(p))
    fit = mb.glmboost(formula, data=data, family=mb.Gaussian(),
                       control=mb.boost_control(mstop=mstop, nu=nu))
    diffs = np.diff(fit.risk_)
    if not np.all(diffs <= 1e-9):
        failures.append((seed, n, p, mstop, nu, diffs.max()))
    assert np.all(diffs <= 1e-9)


check_risk_nonincreasing()
print(f"30 randomly generated (n, p, mstop, nu) combinations checked, "
      f"{len(failures)} failures")


30 randomly generated (n, p, mstop, nu) combinations checked, 0 failures


This extends the fixed-example version: the monotone-risk-decrease
assertion ([](../methodology/theory_properties), Theorem 1) holds
across whatever `(n, p, mstop, nu)` combinations `hypothesis` happens to
draw. If a
future change to the engine broke monotonicity only for, say, `p=1` or a
particular `nu` range, the fixed-seed test in `test_glmboost_risk_nonincreasing`
could miss it while this one would eventually generate that case. When
`hypothesis` finds a failure, it shrinks the failing inputs to a minimal
reproducing example automatically — that minimal case is what you'd debug,
not the original random draw.



## Failing parity tests

A failing parity test means one of three things, in order of likelihood:

1. **A real regression** in the core. Bisect with `git bisect` over the
   failing test; the bad commit is usually obvious. Run the corresponding
   Rust unit test (`cargo test -p mboostrs-core`) — if it also fails, the
   bug is in the core, not the binding.
2. **A documented intentional difference** (see [](parity_and_bugs.ipynb)).
   These are pinned by tests that assert the *mboostrs* behavior, not R's;
   check whether the failing test is one of those.
3. **An R-side quirk** (e.g. the `dNBI` recycling bug,
   [](parity_and_bugs.ipynb)). Reproduce in isolation; if the isolated result conflicts with R's documentation or an independent calculation, record the discrepancy and pin the mboostrs behavior.

Never relax a tolerance to make a parity test pass without understanding the
divergence. If the cause is floating-point path divergence (e.g. the GAMBoost
weight-scaling drift, $\sim 10^{-6}$), document it in [](parity_and_bugs.ipynb)
and *then* set the tolerance to match the observed gap.

## CI wiring

The book-deploy workflow (`.github/workflows/deploy-book.yml`) and the test
workflow run:

- `cargo fmt --check --all`, `cargo clippy --workspace -- -D warnings`,
  `cargo test --workspace`.
- `pytest -m "not r_parity"` (CI has no R).
- `ruff check python/`.
- `myst build --html --execute` for the book (executes every code cell).

Parity tests are local-only by design. The CI suite catches the structural
and internal-correctness regressions; the parity suite catches R-correspondence
regressions for developers who have R installed.

## See also

- [](parity_and_bugs.ipynb) — the documented differences from R, with rationale.
- [](lss_engine.ipynb) — where the finite-diff helper lives and what it pins.

- [](../methodology/families) — the loss, gradient, and response-map formulas
  that the finite-difference test verifies against the analytical gradient.